In [23]:
import pandas
from helpers.data import MinioHelper, CalcHelper
import plotly.express as px

In [24]:
YEARS = ['2023']
SEASON = 'regular'
BUCKET = 'football-warehouse'

In [25]:
helper = MinioHelper(BUCKET)

game_stats_frame = helper.get_statistics(YEARS, SEASON, 'games')
stats_frame = helper.get_statistics(YEARS, SEASON, 'teams')

In [26]:
stats_pivot = stats_frame.pivot_table(index=['team', 'opponent', 'year', 'week'], columns='statistic_name', values='statistic_value', aggfunc='sum', fill_value=0)

In [27]:
stats_pivot = CalcHelper.calculate_efficiency(stats_pivot)

In [28]:
stats_flat = stats_pivot.reset_index()

In [29]:
drop_columns = ['game_id', 'location', 'city', 'state', 'game_date', 'is_conference', 'note', 'line', 'over_under', 'game_type']

home_drops = ['away_team']
home_drops.extend(drop_columns)

away_drops = ['home_team']
away_drops.extend(drop_columns)

home_score_df = game_stats_frame.drop(labels=home_drops, axis=1)
home_score_df = home_score_df.rename(columns={'home_team': 'team', 'home_score': 'points', 'away_score': 'pointsagainst'})

away_score_df = game_stats_frame.drop(labels=away_drops, axis=1)
away_score_df = away_score_df.rename(columns={'away_team': 'team', 'away_score': 'points', 'home_score': 'pointsagainst'})


In [30]:
score_frame = pandas.concat([home_score_df, away_score_df],ignore_index=True)

In [31]:
stats_joined_frame = stats_flat.merge(score_frame, left_on=['team', 'year', 'week'], right_on=['team', 'year', 'week'], how='left')

In [32]:
stats_joined_frame = CalcHelper.calculate_result(stats_joined_frame)
stats_joined_frame.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   team                   544 non-null    object 
 1   opponent               544 non-null    object 
 2   year                   544 non-null    int64  
 3   week                   544 non-null    int64  
 4   attempts               544 non-null    float64
 5   completions            544 non-null    float64
 6   defensivetouchdowns    544 non-null    float64
 7   firstdowns             544 non-null    float64
 8   firstdownspassing      544 non-null    float64
 9   firstdownspenalty      544 non-null    float64
 10  firstdownsrushing      544 non-null    float64
 11  fourthdownattempts     544 non-null    float64
 12  fourthdowncompletions  544 non-null    float64
 13  fumbleslost            544 non-null    float64
 14  interceptions          544 non-null    float64
 15  netpas

In [51]:
fig = px.scatter(stats_joined_frame, x='points', y='rushingyards', color='result')
fig.show()